#### THIS CODE PLOTS THE AMOUNT OF NORMALIZED MM WAVES, COMETO-CENTRIC DISTANCE, MAGNETIC FIELD MODULUS

### It was ploting the waves after the prominence method and not the waves after the peaks method....

### Functions

In [1]:
import datetime 
import numpy as np
import mpmath as mp
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import copy
from sklearn.preprocessing import StandardScaler
%matplotlib qt

import os 
from os import listdir
from os.path import isfile, join
#------------------------------------------------------------------------------------
#This function receives a list with magnetic fields data and returns a list with the values of deltaB/B

def newdeltas(magneticfieldcolumn):
    rolling4=magneticfieldcolumn.rolling(600,center=True).mean() #moving average
    element=[] 
    for index1 in np.arange(0,len(rolling4)): #for every element in the rolling average
        if rolling4[index1]==np.nan:
            element.append(np.nan)
        else:    
            delta=((2*(magneticfieldcolumn[index1]-rolling4[index1]))/rolling4[index1])
            element.append(abs(delta))
            
    return element
#-----------------------------------------------------------------------------------------
#PCAnew returns the sorted eigenvalues and eigenvectors of the covariance matrix calculated with the X data 

def PCAnew(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    check_for_nan_1 = X['Bx'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True:
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        x_std = StandardScaler().fit_transform(data_copy)
        #------------------------------------------------
    
        # features are columns from x_std
        features = x_std.T 
        covariance_matrix = np.cov(features)
        print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#-------------------------------------------------------------------------------------------
#Angles between the vectors

import numpy as np
import numpy.linalg as LA

def angles(a,b): #Returns the angle in degrees and radians
    inner = np.inner(a, b)
    norms = LA.norm(a) * LA.norm(b)

    cos = inner / norms
    rad = np.arccos(np.clip(cos, -1.0, 1.0))
    deg = np.rad2deg(rad)
    
    return deg, rad

#-----------------------------------------------------------------------------------------
#bgmgvector is a list with the background magnetic fields vector
  
#The method consist of an iterating decomposing of the data matrix second by second
#For example if the time used is 3 minutes-->180 seconds. The method will use 90 seconds to the left and 90 seconds to the right (similar to the moving average)
#It should be noted that in this example the first 90 and last 90 points will not have any value 


#This function returns a list of lists that contains the eigenvalues ratios of the PCA method, 
#the angles phi and theta and the biggest deltaB/B value for each iteration

def function1(X,n, newdeltaB, bgmgvector): #n must be an even integer
    element=[]
    nan=[np.nan, np.nan, np.nan, np.nan, np.nan]
    L=X.shape[0] #Size of the data
    for i in np.arange(0,L): #For each data set 
        #The first two ifs are going to tell us if the interval exists and allows the iteration to work, based on the quantity of points 
        if i<(n/2):
            element.append(nan)
        elif (L-i)<=(n/2):
            element.append(nan) 
        else: #If the interval exists
            newdata=X.iloc[np.arange(i-n/2,i+1+n/2),:] #Select the interval
            newdataf=newdata.reset_index(drop=True) 
            newdata2=list(newdeltaB[i] for i in np.arange(int(i-n/2),int(i+1+n/2)))
            method=PCA2(newdataf) #PCA using the data of the interval (this fuction returns the eigenvalues and eigenvectors)
            
            if np.isnan(method[0][0])==False and np.isnan(method[1][0][0])==False: #If the PCA exist
                eigenvector1=method[1][0] #eigenvector associated with the biggest eigenvalue 
                eigenvector2=method[1][2] #eigenvector associated with the smallest eigenvalue
                ratio1=method[0][0]/method[0][1] #Goodness betwen the biggest and the middle eigenvalue
                ratio2=method[0][1]/method[0][2] #Goodness betwen the middle and smallest eigenvalue
                mgvector=bgmgvector[i] #each background magnetic field vector
                theta=angles(eigenvector2,mgvector)[0] #angle in degrees associated with the smallest eigenvalue
                phi=angles(eigenvector1,mgvector)[0] #angle in degrees associated with the biggest eigenvalue
                #If the angle is needed in radians change angles[0] to angles [1]
                
                array=[ratio1, ratio2, theta,phi, max(newdata2)]  ##### MAYBE THIS IS NOT WORKING
                #array=[ratio1, ratio2, theta,phi, newdeltaB[i]]
                element.append(array)
            else:
                element.append(nan)
    return element

#------------------------------------------------------------------------------------------------------
def getting_day(data_plasma, year, month, day):
    
    time_plasma=pd.to_datetime(data_plasma['t_utc'])
    #year=2015
    mask = time_plasma.dt.year == int(year)
    include = data_plasma[mask]
    time_plasma2=pd.to_datetime(include['t_utc'])
    #month='06'
    mask2=time_plasma2.dt.month == int(month)
    include2 = include[mask2]
    time_plasma3=pd.to_datetime(include2['t_utc'])
    #day='07'
    mask3=time_plasma3.dt.day == int(day)
    include3=include2[mask3]

    time_plasma4=pd.to_datetime(include3['t_utc'])
    
  
    return time_plasma4, include3['npl']
#------------------------------------------------------------------------------------------------------
def get_directory(folder):
    files= [a for a in listdir(folder) if isfile(join(folder,a))]
    
    return files

#------------------------------------------------------------------------------------------------------
#This function receives a Mirror Mode Waves Candidates DataFrame and returns its time intervals and how many MM are in that day

def intervals(element, n, array1):  
    #modifying the MM table
    element['tvalue'] = element.index
    element['delta'] = (element['tvalue']-element['tvalue'].shift()).fillna(0)
    zx=element.copy()
    zx2=zx.reset_index(drop=True)
    deltas=zx2['delta']
    deltas=deltas.values.tolist()
    indexA=zx2['Index']
    
    #Empty Dataframes
    my_df=pd.DataFrame()
    my_df2=pd.DataFrame()
    
    if len(element['tvalue'])!=0: #If we have MM waves
    
        #LIMIT CONDITIONS
        limits=[]
        limits2=[]
        for i in np.arange(0,len(zx2)):
            if deltas[i]>(n/2) or deltas[i]==0: #Same event
                limits.append(indexA[i])
                if i!=0:
                    limits2.append(indexA[i-1])
        limits2.append(indexA[len(deltas)-1])   
        
    
        if len(limits)!=1: #If there is not only one MM
    
            for i in np.arange(0,len(limits)): 
                index1=limits[i]
                index2=limits2[i]
                my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True) 
    
        else: #If there is only one MM
            index1=limits[0]
            index2=limits2[0]
            large=len(deltas)
            my_df=my_df.append({'Beginning' : array1[index1-(n/2)], 'End' : array1[index2+(n/2)], 'Index1' : index1, 'Index2': index2}, ignore_index=True)
    else: #If we have not MM
        my_df=my_df
        limits=[]
        limits2=[]


    return my_df, len(my_df), limits,limits2,


def PCA2(X):
    
    #The covariance matrix does not work with NaN values
    #------------------------------------------------
    #print(X)
    check_for_nan_1 = X['Bx'].isnull()
    check_for_nan_2 = X['By'].isnull()
    check_for_nan_3 = X['Bz'].isnull()
    data_copy = X.copy()
    for i in np.arange(0,len(X)):
        if check_for_nan_1[i]==True or check_for_nan_2[i]==True or check_for_nan_3[i]==True :
            data_copy.drop(i, axis=0, inplace=True)
    #After this we droped all the nan values in the specific interval
    #print(data_copy)
    #-----------------------------------------------------------------------------
    if len(data_copy)!=1 and len(data_copy)!=2 and len(data_copy)!=3 and X.dropna().empty==False: #If the interval does not contain nan numbers
    
        covariance_matrix=data_copy.cov()
        #print(covariance_matrix)
        #Eigen Vectors and Eigen Values from Covariance Matrix
        eig_vals, eig_vecs = np.linalg.eig(covariance_matrix)
    
        #In order to print in descending order the vectors...
        vec=eig_vecs.T
        sortedd=sorted(eig_vals, reverse=True)
        neww=[]
    
        for index1 in np.arange(0,3):  
             for index2 in np.arange(0,3):
                if eig_vals[index2]==sortedd[index1]:
                    neww.append(vec[index2])
    else:
        sortedd=[np.nan, np.nan, np.nan]
        neww=[[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan],[np.nan, np.nan, np.nan]]

    return sortedd, neww  #In order to compute easier it is used the transpose of the eig_vecs 

#---------------------------------------------------------------------------------------------------------------

In [2]:
### Outgassing

In [3]:
outgassing="C:/Users/Ariel/Desktop/ESA/Data_GasProduction/gasproduction.txt"

title2=['Timestamp','Neutral gas density','Derived gas production rate']
path= str(outgassing)
data1= pd.read_table(path, header=None, names=title2, sep='\s+', engine='python')
 
#Filtering
outgassing_copy=data1.copy() 
for j in np.arange(0,len(outgassing_copy)):
    if outgassing_copy['Derived gas production rate'][j]<1000000: 
        outgassing_copy['Timestamp'][j]=np.nan
        
for j in np.arange(0,len(outgassing_copy)):
    if pd.isnull(outgassing_copy['Timestamp'][j])==True:
        outgassing_copy.drop(j, axis=0, inplace=True)
        
        
data_gas=outgassing_copy.reset_index(drop=True)

C:\Users\Ariel\AppData\Local\Temp\ipykernel_8944\3139886205.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outgassing_copy['Timestamp'][j]=np.nan


In [4]:
def getting_day2(data_gas, year, month, day):
    
    time_gas=pd.to_datetime(data_gas['Timestamp'])
    #year=2015
    mask = time_gas.dt.year == int(year)
    include = data_gas[mask]
    time_gas2=pd.to_datetime(include['Timestamp'])
    #month='06'
    mask2=time_gas2.dt.month == int(month)
    include2 = include[mask2]
    time_gas3=pd.to_datetime(include2['Timestamp'])
    #day='07'
    mask3=time_gas3.dt.day == int(day)
    include3=include2[mask3]

    time_gas4=pd.to_datetime(include3['Timestamp'])
    
  
    return time_gas4, include3['Derived gas production rate']

## Entire year plot

In [5]:
#data folder

MM_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/MM all mission with data/new_plots_prominence_withoutnegativevalues_CHECKED"
#---------------------------------------------------------

list_of_files_MM=get_directory(MM_folder)

#List with the paths
newlist2=[]
for item in list_of_files_MM:
    newlist2.append(MM_folder+'/'+str(item))
print(newlist2)

events=[] 
for i in np.arange(0, len(newlist2)):
        title2=['Days','Normalized_Data', 'Outgassing','Distance_to_the_comet','Magnetic_field_strength']
        path= str(newlist2[i])
        data1= pd.read_table(path, header=None, names=title2, sep='\t')
        #data1= pd.read_table(path)
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        #data2=data2.reset_index(drop=True)
        events.append(data2)
        
        
#MM waves folder

MM_folder2="C:/Users/Ariel/Desktop/JUPYTER/Resultados/peakswithoutnegativevaluesnpl_CHECKED/total_data"
#---------------------------------------------------------

list_of_files_MM2=get_directory(MM_folder2)

#List with the paths
newlist3=[]
for item in list_of_files_MM2:
    newlist3.append(MM_folder2+'/'+str(item))
#print(newlist3)

amountofwaves=[] 
for i in np.arange(0, len(newlist3)):
        title2=['Beginning','End', 'Index1','Index2','Pearson']
        path= str(newlist3[i])
        data1= pd.read_table(path, header=None, names=title2,sep='\t')
        #data1= pd.read_table(path)
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        #data2=data2.reset_index(drop=True)
        amountofwaves.append(len(data2))        

['C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/MM all mission with data/new_plots_prominence_withoutnegativevalues_CHECKED/data2_201411', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/MM all mission with data/new_plots_prominence_withoutnegativevalues_CHECKED/data2_201412', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/MM all mission with data/new_plots_prominence_withoutnegativevalues_CHECKED/data2_201501', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/MM all mission with data/new_plots_prominence_withoutnegativevalues_CHECKED/data2_201502', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/MM all mission with data/new_plots_prominence_withoutnegativevalues_CHECKED/data2_201503', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/MM all mission with data/new_plots_prominence_withoutnegativevalues_CHECKED/data2_201504', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/MM all mission with data/new_plots_prominence_withoutnegativevalues_CHECKED

## We need the plasma data in order to see the error bars

In [6]:
#MM waves folder

#MM_folder="C:/Users/Ariel/Desktop/JUPYTER/Mirror_Mode_Waves_Linear_Regression/DATA"
MM_folder="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED"
#---------------------------------------------------------

list_of_files_MM=get_directory(MM_folder)

#List with the paths
newlist2=[]
for item in list_of_files_MM:
    newlist2.append(MM_folder+'/'+str(item))
print(newlist2)
events2=[] #Amount of MM waves per day
for i in np.arange(0, len(newlist2)):
        title2=['Days','Normalized_Data','Data_Plasma','Waves']
        path= str(newlist2[i])
        data1= pd.read_table(path, header=None, names=title2, sep='\t', engine='python')
        #data1= pd.read_table(path)
        data2=data1.copy()
        data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row
        #data2=data2.reset_index(drop=True)
        events2.append(data2)

['C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201411', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201412', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201501', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201502', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201503', 'C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201504', 'C:/Users/Ariel

In [7]:
list_of_normalized_data=[]
list_of_data_plasma=[]
list_of_waves=[]
for i in np.arange(0,len(newlist2)):
    for j in np.arange(0,len(events2[i])):
        list_of_normalized_data.append(events2[i]['Normalized_Data'][j+1])
        list_of_data_plasma.append(events2[i]['Data_Plasma'][j+1])
        list_of_waves.append(events2[i]['Waves'][j+1])
     
    
data_plasma=list_of_data_plasma.copy()
data_plasma_ses=[]    
for i in np.arange(0,len(data_plasma)):
    if type(data_plasma[i])!=str:
        data_plasma_ses.append(0)
    elif type(data_plasma[i])==str:
        data_plasma_ses.append(float(data_plasma[i]))

data_plasma2=[]
for i in np.arange(0,len(data_plasma)):
    data_plasma2.append(data_plasma_ses[i])
#print(data_plasma2)    

In [8]:
list_of_normalized_data=[]
list_of_outgassing=[]
list_of_distance=[]
list_of_magnetic=[]
for i in np.arange(0,len(newlist2)):
    for j in np.arange(0,len(events[i])):
        list_of_normalized_data.append(events[i]['Normalized_Data'][j+1]) #It is fine j+1
        list_of_outgassing.append(events[i]['Outgassing'][j+1])
        list_of_distance.append(events[i]['Distance_to_the_comet'][j+1])
        list_of_magnetic.append(events[i]['Magnetic_field_strength'][j+1])

        
x=np.arange(1,len(list_of_outgassing)+1)        

In [20]:
print(events[0])

   Days Normalized_Data              Outgassing Distance_to_the_comet  \
1     1             NaN  1.5562863752605983e+26     33.79606890988813   
2     2             NaN  1.6565333688293382e+26     33.43143155020476   
3     3             NaN   1.446983031802119e+26     32.62679179211197   
4     4             NaN   1.094376984016678e+26    31.753202003354904   
5     5             NaN   9.672079355509348e+25    30.909911915596997   
6     6             0.0  1.0561547116052814e+26      30.2065929160041   
7     7             0.0   1.852015786414564e+26    29.720651720235946   
8     8             NaN  2.5193436938202314e+26    29.464472395770148   
9     9             0.0   3.500828257777775e+26    29.487937427472385   
10   10             0.0  4.1962080547752874e+26    29.686441895247427   
11   11             NaN   3.641017639751554e+26    29.817887594260366   
12   12             NaN   3.778095868005739e+26    21.941120467617402   
13   13             NaN  2.4780627687626784e+26    

In [9]:
#----------------------------
list_of_normalized_data2=[]
for i in np.arange(0,len(list_of_normalized_data)):
    list_of_normalized_data2.append(float(list_of_normalized_data[i]))
#print(list_of_normalized_data2)

#----------------------------
data_outgassing=list_of_outgassing.copy()
data_outgassing_ses=[]    
for i in np.arange(0,len(data_outgassing)):
    if type(data_outgassing[i])!=str:
        data_outgassing_ses.append(np.nan)
    elif type(data_outgassing[i])==str:
        data_outgassing_ses.append(float(data_outgassing[i]))

list_of_outgassing2=[]
for i in np.arange(0,len(data_outgassing)):
    list_of_outgassing2.append(data_outgassing_ses[i])
#print(list_of_outgassing2)
#------------------------------
list_of_distance2=[]
for i in np.arange(0,len(list_of_distance)):
    if type(list_of_distance[i])!=str:
        list_of_distance2.append(np.nan)
    elif type(list_of_distance[i])==str:
        list_of_distance2.append(float(list_of_distance[i]))

#print(list_of_distance2)
#------------------------------
list_of_magnetic2=[]
for i in np.arange(0,len(list_of_magnetic)):
    if type(list_of_magnetic[i])!=str:
        list_of_magnetic2.append(np.nan)
    elif type(list_of_magnetic[i])==str:
        list_of_magnetic2.append(float(list_of_magnetic[i]))

#print(type(list_of_magnetic2[0]))

#--------------------------------
import math

outgassing_log=[]

for i in np.arange(0,len(list_of_outgassing2)):
    #print(i)
    if np.isnan(list_of_outgassing2[i])==True:
        outgassing_log.append(np.nan)
    
    else:
        outgassing_log.append(math.log(list_of_outgassing2[i],10))
        


In [10]:
list_of_normalized_data3=list_of_normalized_data2.copy()
list_of_normalized_data4=np.zeros(len(x))

for i in np.arange(0,len(list_of_normalized_data3)):
    if list_of_normalized_data3[i]<0 or list_of_normalized_data3[i]>0:
        list_of_normalized_data4[i]=list_of_normalized_data3[i]
#print(list_of_normalized_data4)     

In [11]:
errors=[]

for i in np.arange(0,len(list_of_data_plasma)):
    
    if int(list_of_data_plasma[i])==0:
        Error=0
        
    else:    
        Error=1/int(list_of_data_plasma[i])
    errors.append(Error)
    
errors2=errors.copy() #Not considering the values with 0

for i in np.arange(0,len(list_of_normalized_data)):
    if np.isnan(float(list_of_normalized_data[i]))==True:
        errors2[i]=np.nan




In [12]:
print(list_of_normalized_data[316])
print(list_of_data_plasma[316])
print(errors2[316])

1.2017786323759164e-05
83210
1.2017786323759164e-05


## OKK I TRIED TO PLOT THE ERROR BARS OF THIS PLOT BUT WE HAVE SOME DAYS WITH ONLY 1 WAVE. THUS, IT DOES NOT MAKES SENSE TO SEE THE ERROR

In [13]:
barWidth=0.3
barWidthnormalized=0.3

'Train wave found at x[102-1], mirror mode event used in the paper at x[290-1]' #agosto 17 2015

#FIRST PLOT
plt.figure()
ax7=plt.subplot(414)
figsize=(20,4)
#barWidth=0.3
#barWidthnormalized=0.3
#plt.bar(x,list_of_magnetic2,color='red', width=barWidthnormalized, label='Magnetic field modulus')   
plt.plot(x,list_of_magnetic2,"r-", label='Magnetic field modulus')  
plt.text(0.01, 0.75, "D)", fontweight="bold", fontsize=13 ,transform=ax7.transAxes)
plt.ylabel('|B| (nT)',fontsize=13)
#specify x-axis locations
x_ticks = [1,31,62,90,121,151,182,212,243,274,304,335,365,396,427,456]
#specify x-axis labels
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
plt.xticks(ticks=x_ticks, labels=x_labels,fontsize=13)

plt.bar(x[290-1],list_of_magnetic2[290-1] ,color='black', width=barWidth)
plt.bar(x[102-1],list_of_magnetic2[102-1] ,color='black', width=barWidth)
#plt.axvline(x[290], c='black')
#plt.axvline(x[101], c='black')


#SECOND PLOT

ax1=plt.subplot(411,sharex=ax7)
figsize=(20,4)
plt.bar(x,list_of_normalized_data4 ,color='purple', width=barWidth, label='Amount of normalized data')
#plt.bar(x[290],list_of_normalized_data4[290] ,color='black', width=barWidth, label='Amount of normalized data')
#plt.bar(x[101],list_of_normalized_data4[101] ,color='black', width=barWidth, label='Amount of normalized data')
plt.ylabel('%',fontsize=13)
plt.text(0.01, 0.75, "A)", fontweight="bold", fontsize=13 ,transform=ax1.transAxes)
plt.bar(x[102-1],list_of_normalized_data4[102-1] ,color='black', width=barWidth)
plt.bar(x[290-1],list_of_normalized_data4[290-1] ,color='black', width=barWidth)

plt.yscale('log')
ax1.set_yticks([1e-6,1e-5,1e-4])
plt.setp(ax1.get_xticklabels(), fontsize=6)
plt.setp(ax1.get_xticklabels(), visible=False)

import pylab as p
plt.annotate('', ha = 'center', va = 'bottom',
xytext = (x[285], list_of_normalized_data4[285]+0.00005),xy = (x[290-1]-0.5, list_of_normalized_data4[290-1]+0.000005),arrowprops = {'width': 0.1,'headwidth':7,'headlength':7,'facecolor' : 'black'},)

import pylab as p
plt.annotate('', ha = 'center', va = 'bottom',
xytext = (x[106]-15, list_of_normalized_data4[101]-0.00005),xy = (x[100]-1, list_of_normalized_data4[101]-0.00001),arrowprops = {'width': 0.1,'headwidth':7,'headlength':7,'facecolor' : 'black'},)


y_errormin=errors2
y_errormax=errors2
y_error =[y_errormin, y_errormax]


#plt.errorbar(x, list_of_normalized_data4, yerr = y_error, fmt ='o')



#THIRD PLOT

ax2=plt.subplot(412,sharex=ax7)
figsize=(20,4)
plt.plot(x,list_of_outgassing2, color='blue', label='Gas production rates log scale') 
plt.ylabel('Q ($s^{-1}$)',fontsize=13)
plt.yscale('log')
plt.setp(ax2.get_xticklabels(), visible=False)
plt.text(0.01, 0.75, "B)", fontweight="bold", fontsize=13 ,transform=ax2.transAxes)
#ax2.set_yticks([1e+27,1e+29])
#import pylab as p
#plt.annotate('', ha = 'center', va = 'bottom', 
#xytext = (x[295], list_of_outgassing2[290]+0.0001),xy = (x[290], list_of_outgassing2[290]+0.00001),arrowprops = {'width': 0.1,'headwidth':7,'headlength':7,'facecolor' : 'black'},)

#plt.annotate('', ha = 'center', va = 'bottom', 
#xytext = (x[106], list_of_outgassing2[101]+0.0001),xy = (x[101], list_of_outgassing2[101]+0.00001),arrowprops = {'width': 0.1,'headwidth':7,'headlength':7,'facecolor' : 'black'},)


#plt.axvline(x[290], c='black')
#plt.axvline(x[101], c='black')
plt.bar(x[290-1],list_of_outgassing2[290-1] ,color='black', width=barWidth)
plt.bar(x[102-1],list_of_outgassing2[102-1] ,color='black', width=barWidth)

#FOURTH PLOT

ax2=plt.subplot(413,sharex=ax7)
figsize=(20,4)
plt.plot(x,list_of_distance2, color='black', label='Distance to the comet')  
ax2.set_ylim(0,500)
plt.ylabel('r (km)',fontsize=13)
plt.setp(ax2.get_xticklabels(), visible=False)
plt.text(0.01, 0.75, "C)", fontweight="bold", fontsize=13 ,transform=ax2.transAxes)

plt.bar(x[290-1],list_of_distance2[290-1] ,color='black', width=barWidth)
plt.bar(x[102-1],list_of_distance2[102-1] ,color='black', width=barWidth)
#plt.axvline(x[290], c='black')
#plt.axvline(x[101], c='black')

<BarContainer object of 1 artists>

### Same plot but with plasma density

In [14]:
barWidth=0.3
barWidthnormalized=0.3

'Train wave found at x[102-1], mirror mode event used in the paper at x[290-1]' #agosto 17 2015

#FIFTH PLOT
plt.figure()
ax7=plt.subplot(515)

figsize=(20,4)
#barWidth=0.3
#barWidthnormalized=0.3
#plt.bar(x,list_of_magnetic2,color='red', width=barWidthnormalized, label='Magnetic field modulus')   
plt.plot(x,list_of_magnetic2,"r-", label='Magnetic field modulus')  
#plt.text(0.01, 0.75, "E)", fontweight="bold", fontsize=13 ,transform=ax7.transAxes)
plt.ylabel('|B| (nT)',fontsize=13)
#specify x-axis locations
x_ticks = [1,31,62,90,121,151,182,212,243,274,304,335,365,396,427,456]
#specify x-axis labels
x_labels = ['2014-Nov', '','2015-Jan','','2015-Mar','','2015-May','','2015-Jul','','2015-Sep','','2015-Nov','','2016-Jan',''] 
plt.xticks(ticks=x_ticks, labels=x_labels,fontsize=13)
#fig1.align_ylabels()
ax7.get_yaxis().set_label_coords(-0.05,0.5)

plt.text(-0.10, 0.42, "E)", fontweight="bold", fontsize=13 ,transform=ax7.transAxes)

#plt.bar(x[290-1],list_of_magnetic2[290-1] ,color='black', width=barWidth)
#plt.bar(x[102-1],list_of_magnetic2[102-1] ,color='black', width=barWidth)
plt.axvline(x[290-1],ls=':', c='purple')
plt.axvline(x[102-1],ls=':', c='purple')
#plt.axvline(x[290], c='black')
#plt.axvline(x[101], c='black')


#FIRST PLOT

ax1=plt.subplot(511,sharex=ax7)
#fig2,ax1=plt.subplots(511,sharex=ax7)
figsize=(20,4)
plt.bar(x,list_of_normalized_data4 ,color='purple', width=barWidth, label='Amount of normalized data')
#plt.bar(x[290],list_of_normalized_data4[290] ,color='black', width=barWidth, label='Amount of normalized data')
#plt.bar(x[101],list_of_normalized_data4[101] ,color='black', width=barWidth, label='Amount of normalized data')
plt.ylabel(r'$\Omega$',fontsize=13)
#plt.text(0.01, 0.75, "A)", fontweight="bold", fontsize=13 ,transform=ax1.transAxes)
plt.text(-0.10, 0.42, "A)", fontweight="bold", fontsize=13 ,transform=ax1.transAxes)
plt.bar(x[102-1],list_of_normalized_data4[102-1] ,color='black', width=barWidth)
plt.bar(x[290-1],list_of_normalized_data4[290-1] ,color='black', width=barWidth)

plt.yscale('log')
ax1.set_yticks([1e-6,1e-5,1e-4])
plt.setp(ax1.get_xticklabels(), fontsize=6)
plt.setp(ax1.get_xticklabels(), visible=False)

ax1.get_yaxis().set_label_coords(-0.05,0.5)


import pylab as p
plt.annotate('', ha = 'center', va = 'bottom',
xytext = (x[285], list_of_normalized_data4[285]+0.00005),xy = (x[290-1]-0.5, list_of_normalized_data4[290-1]+0.000005),arrowprops = {'width': 0.1,'headwidth':7,'headlength':7,'facecolor' : 'black'},)

import pylab as p
plt.annotate('', ha = 'center', va = 'bottom',
xytext = (x[106]-15, list_of_normalized_data4[101]-0.00005),xy = (x[100]-1, list_of_normalized_data4[101]-0.00001),arrowprops = {'width': 0.1,'headwidth':7,'headlength':7,'facecolor' : 'black'},)


y_errormin=errors2
y_errormax=errors2
y_error =[y_errormin, y_errormax]


#SECOND PLOT

ax2=plt.subplot(512,sharex=ax7)
#fig3,ax2=plt.subplots(512,sharex=ax7)
figsize=(20,4)
barWidth=0.3
#ax2.set_ylim(0,500)
barWidthnormalized=0.3
plt.bar(x,data_plasma2,color='green', width=barWidth, label='Amount of plasma density data')   
#plt.text(0.01, 0.75, "B)", fontweight="bold", fontsize=13 ,transform=ax2.transAxes)
plt.text(-0.10, 0.42, "B)", fontweight="bold", fontsize=13 ,transform=ax2.transAxes)
#ax2.set_yticks([0,30000,60000,80000])
ax2.set_yticks([1e3,1e5])
plt.ylabel('PDDs',fontsize=13)
plt.yscale('log')
#plt.setp(ax2.get_xticklabels(), fontsize=6)
#plt.title('Mirror mode waves',fontsize=15)
#plt.legend()
plt.setp(ax2.get_xticklabels(), visible=False)
plt.bar(x[290-1],data_plasma2[290-1] ,color='black', width=barWidth)
plt.bar(x[102-1],data_plasma2[102-1] ,color='black', width=barWidth)

ax2.get_yaxis().set_label_coords(-0.05,0.5)


#THIRD PLOT

#fig4,ax2=plt.subplots(513,sharex=ax7)
ax2=plt.subplot(513,sharex=ax7)
figsize=(20,4)
plt.plot(x,list_of_outgassing2, color='blue', label='Gas production rates log scale') 
#plt.text(0.01, 0.75, "C)", fontweight="bold", fontsize=13 ,transform=ax2.transAxes)
plt.text(-0.10, 0.42, "C)", fontweight="bold", fontsize=13 ,transform=ax2.transAxes)
plt.ylabel('Q ($s^{-1}$)',fontsize=13)
plt.yscale('log')
plt.setp(ax2.get_xticklabels(), visible=False)
#ax2.set_yticks([1e+27,1e+29])
#import pylab as p
#plt.annotate('', ha = 'center', va = 'bottom', 
#xytext = (x[295], list_of_outgassing2[290]+0.0001),xy = (x[290], list_of_outgassing2[290]+0.00001),arrowprops = {'width': 0.1,'headwidth':7,'headlength':7,'facecolor' : 'black'},)

#plt.annotate('', ha = 'center', va = 'bottom', 
#xytext = (x[106], list_of_outgassing2[101]+0.0001),xy = (x[101], list_of_outgassing2[101]+0.00001),arrowprops = {'width': 0.1,'headwidth':7,'headlength':7,'facecolor' : 'black'},)


#plt.axvline(x[290], c='black')
#plt.axvline(x[101], c='black')
#plt.bar(x[290-1],list_of_outgassing2[290-1] ,color='black', width=barWidth)
#plt.bar(x[102-1],list_of_outgassing2[102-1] ,color='black', width=barWidth)

#plt.vlines(x=39.25, ymin=25, ymax=150, colors='black', ls=':', lw=2, label='vline_single - partial height')
#plt.vlines(x[290-1],list_of_outgassing2[290-1], colors='purple', ls='--', lw=2, label='vline_multiple - full height')

plt.axvline(x[290-1],ls=':', c='purple')
plt.axvline(x[102-1],ls=':', c='purple')

ax2.get_yaxis().set_label_coords(-0.05,0.5)


#FOURTH PLOT

ax2=plt.subplot(514,sharex=ax7)
#fig5,ax2=plt.subplots(514,sharex=ax7)
figsize=(20,4)
plt.plot(x,list_of_distance2, color='black', label='Distance to the comet')  
ax2.set_ylim(0,500)
plt.ylabel('r (km)',fontsize=13)
plt.setp(ax2.get_xticklabels(), visible=False)

#plt.text(0.01, 0.75, "D)", fontweight="bold", fontsize=13 ,transform=ax2.transAxes)
plt.text(-0.10, 0.42, "D)", fontweight="bold", fontsize=13 ,transform=ax2.transAxes)
#plt.bar(x[290-1],list_of_distance2[290-1] ,color='black', width=barWidth)
#plt.bar(x[102-1],list_of_distance2[102-1] ,color='black', width=barWidth)
plt.axvline(x[290-1],ls=':', c='purple')
plt.axvline(x[102-1],ls=':', c='purple')
#plt.axvline(x[290], c='black')
#plt.axvline(x[101], c='black')
ax2.get_yaxis().set_label_coords(-0.05,0.5)



#FIFTH PLOT

#ax2=plt.subplot(514, sharex=ax7)
#figsize=(20,4)
#plt.plot(x,list_of_data_plasma, color='green', label='Plasma density')
#ax2.set_ylim(0,500)
#plt.ylabel('r (km)',fontsize=13)
#plt.setp(ax2.get_xticklabels(), visible=False)






In [15]:
import pylab as p
#p.arrow( x, y, dx, dy, **kwargs )
p.arrow( 0.5, 0.8, 0.0, -0.2, fc="k", ec="k",head_width=0.0000005, head_length=0.000001 )
p.show()

In [16]:

import numpy as np
import matplotlib.pyplot as plt
X = np.linspace(-3.9, 5, 1019)
Y = .24 * (X + 3.8) * (X + 0.9) * (X - 1.8)
plt.annotate('Brackmard minimum',
ha = 'center', va = 'bottom',
xytext = (-1.8, 3.5),xy = (0.69, - 1.7),arrowprops = {'facecolor' : 'black'})
plt.plot(X, Y)
plt.show()

In [17]:
import matplotlib.pyplot as plt

#define x and y
xx = [1, 4, 10]
y = [5, 11, 27]

#create plot of x and y
plt.plot(xx, y)

#specify x-axis locations
x_ticks = [1, 2, 6, 10]

#specify x-axis labels
x_labels = ['2014-Nov', '2014-Dec', '2014-Jan', 10] 

#add x-axis values to plot
plt.xticks(ticks=x_ticks, labels=x_labels, rotation=90)

([<matplotlib.axis.XTick at 0x1a3daf22a10>,
 [])

In [18]:
#IMPORTAR LOS DATOS ACA:

folder="C:/Users/Ariel/Desktop/JUPYTER/peaks/peaks_201602/prominence"
directory= os.scandir(folder)
list_of_files=get_directory(folder)

newlist=[]
for item in list_of_files:
    newlist.append(folder+'/'+str(item))
print(newlist)


pir=[]
for item in newlist:
    #Importing data
    titles=['Beginning', 'End', 'Index1', 'Index2', 'Pearson']
    data= pd.read_table(str(item), header=None, names=titles, sep='\t')
    data2=data.iloc[np.arange(1,len(data)),:] #delete the first row
    data2=data2.reset_index(drop=True)
    pir.append(len(data2))

print(pir)  

print(sum(pir))

FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: 'C:/Users/Ariel/Desktop/JUPYTER/peaks/peaks_201602/prominence'

In [ ]:
done

In [ ]:
#THIS SECTION RETURNS ALL DATA NEEDED FOR EVERY MONTH (OUTGASSING, DISTANCE TO THE COMET AND MAGNETIC FIELD STRENGTH)

cantidad_de_dias=29
mes=2
folder="C:/Users/Ariel/Desktop/ESA/Data_Rosetta/2016_months/2" 
MM_path="C:/Users/Ariel/Desktop/JUPYTER/Resultados/iteration2/Mirror Mode Waves Per Month/Mirror_Mode_Waves_Using_Prominence_method_notnegativevalues_CHECKED/Data_201602" #CAMBIAR
año=2016

directory= os.scandir(folder)
list_of_files=get_directory(folder)
x=np.arange(1,cantidad_de_dias+1)

#NORMALIZED DATA

title2=['Days','Normalized_Data','Data_Plasma','Waves']
#data1= pd.read_table(MM_path, header=None, names=title2, sep='\s+', engine='python') #ASEGURARSE QUE FUNCIONA BIEN USANDO EL /t
data1= pd.read_table(MM_path, header=None, names=title2, sep='\t')
data2=data1.copy()
data2=data2.iloc[np.arange(1,len(data2)),:] #delete the first row

normalized_values=data2['Normalized_Data'].copy()

#----------------------------------------------------------------------------------------------------------------------------------
#OUTGASSING

outgassing_mean_values=[]
for i in x:
    gas_production_rate_of_that_day=getting_day2(data_gas,año, mes, i)
    mean_value=[]
    
    if len(gas_production_rate_of_that_day[0])==0:
        #New dataframe for the gas production rate 
        a1={'Timestamp':  [], 'Derived gas production rate': []}
        gas_production_rate_of_that_day= pd.DataFrame (a1, columns = ['Timestamp','Derived gas production rate'])
        mean_value.append(np.nan)
    else:
        #New dataframe for the gas production rate 
        a1={'Timestamp':  gas_production_rate_of_that_day[0], 'Derived gas production rate': gas_production_rate_of_that_day[1]}
        gas_production_rate_of_that_day= pd.DataFrame (a1, columns = ['Timestamp','Derived gas production rate'])
        
        mean_value.append(np.mean(gas_production_rate_of_that_day['Derived gas production rate']))
        
        #WE ARE NOT GOING TO FILL THE DATA GAPS BECAUSE WE WILL TAKE THE MEAN VALUE FOR EACH DAY
        
        #Filling the data gaps
        #data_plasma_of_that_day.Dates_and_Hours = data_plasma_of_that_day.Dates_and_Hours.dt.round('1s') #round to one second for simplicity
        #if data_plasma_of_that_day.shape[0] < 86400: # if the number of datapoints is lower than one day:
            #print('Data gaps detected, padding array....')
            #data2 = data_plasma_of_that_day.rename(index=(data_plasma_of_that_day['Dates_and_Hours']-data_plasma_of_that_day.iat[0,0].round('1d')).dt.seconds) # we will index the file new, according to the number of seconds of the data point since the start of the day
            #data2 = data2.reindex(range(0,86400)) # now we just fill in the missing values
            #newt = pd.date_range(start = data_plasma_of_that_day.iat[0,0].round('1d'), periods = 86400, freq = '1s').values # new time array
            #data2['Dates_and_Hours'] = newt # now fill in the times so there is no NaT
            #data_plasma_of_that_day = data2 
            #del(data2)
        #elif data_plasma_of_that_day.shape[0] > 86400:
            #error('Data file is too long, probably need to debug the code again....')
            
    outgassing_mean_values.append(mean_value[0])   
    
#----------------------------------------------------------------------------------------------------------------------------------------------------------------

#DISTANCE TO THE COMET and MAGNETIC FIELD STRENGTH

newlist=[]
for item in list_of_files:
    newlist.append(folder+'/'+str(item))
print(newlist)


magnetic_mean_values=[]
distance_mean_values=[]
for item in newlist:
    #Importing data
    titles=['Dates_and_Hours', '?', 'x', 'y', 'z', 'Bx', 'By', 'Bz', 'flag']
    data= pd.read_table(str(item), header=None, names=titles, sep='\s+', parse_dates=['Dates_and_Hours'])
    
    if len(data)==1:
        magnetic_mean_values.append(np.nan)
        distance_mean_values.append(np.nan)
    else:    
        bmodulus=[]
        distance=[]
        for i in np.arange(0,len(data)):
            Bpoint=(data['Bx'][i]**2+data['By'][i]**2+data['Bz'][i]**2)**(1/2)
            bmodulus.append(Bpoint)
            
            Dpoint=(data['x'][i]**2+data['y'][i]**2+data['z'][i]**2)**(1/2)
            distance.append(Dpoint)
        
        magnetic_mean_values.append(np.mean(bmodulus))
        distance_mean_values.append(np.mean(distance))
        
 


 #SAVING DATA

import csv

#Creating a new dataframe in order to establish the anticorrelation
a2={ 'Days': x, 'Normalized_Data': normalized_values ,'Outgassing': outgassing_mean_values, 'Distance_to_the_comet': distance_mean_values, 'Magnetic_field_strength': magnetic_mean_values }
data_per_month= pd.DataFrame(a2,columns = ['Days','Normalized_Data', 'Outgassing','Distance_to_the_comet','Magnetic_field_strength'])
data_per_month.to_csv('data2'+'_'+str(mes)+'_'+str(año), index=False, sep='\t')